In [9]:
#pip install nbimporter

from transformers import AutoTokenizer, AutoModel
import torch
import nbimporter
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
import copy



In [10]:
all_party_textes_df = pd.read_parquet("all_parties_text_combined_prep_classified.parquet")
labled_theses_df = pd.read_parquet('wahlomat_thesen_prep.parquet')
Labled_theses_classification_df = pd.read_parquet('wahlomaten_thesen_positionen_classified.parquet')

labled_theses_df['Predicted_Class'] = Labled_theses_classification_df['Predicted_Class']
del Labled_theses_classification_df


In [36]:
all_party_textes_df

,Party,Chapter,These,Text,Text_no_stopwords,These_no_stopwords,Text_lemmatized,These_lemmatized,These_unique_words,These_Word_Count,Text_Word_Count,Total_Word_Count,Predicted_Class
3,SPD,1. Wir kämpfen für neues Wachstum und sichere ...,Wir wollen Energiepreise senken und zentrale E...,"Wir brauchen bezahlbare Energie, sonst geht un...",brauchen bezahlbare energie geht unternehmen d...,energiepreise senken zentrale erfolgsbranchen ...,brauchen bezahlbar Energie gehen Unternehmen D...,Energiepreise senken zentral erfolgsbranche st...,Energiepreise senken zentral erfolgsbranche st...,9,659,668,402 - Incentives
4,SPD,1. Wir kämpfen für neues Wachstum und sichere ...,"Wir wollen, dass Unternehmen die besten Rahmen...",Wir können etwas tun für unsere Unternehmen un...,tun unternehmen wirtschaftsstandort schaffen d...,unternehmen besten rahmenbedingungen deutschla...,tun Unternehmen Wirtschaftsstandort schaffen D...,Unternehmen gut Rahmenbedingung Deutschland in...,Unternehmen gut Rahmenbedingung Deutschland in...,13,394,407,402 - Incentives
5,SPD,1. Wir kämpfen für neues Wachstum und sichere ...,Wir wollen Bürokratie abbauen und Verfahren be...,Damit unsere Wirtschaft wieder schnell in Schw...,wirtschaft schnell schwung kommt müssen viele ...,bürokratie abbauen verfahren beschleunigen,Wirtschaft schnell Schwung kommen müssen viele...,Bürokratie abbauen Verfahren beschleunigen,Bürokratie abbauen Verfahren beschleunigen,7,266,273,303 - Governmental and Administrative Efficiency
6,SPD,1. Wir kämpfen für neues Wachstum und sichere ...,"Wir wollen Innovationen ermöglichen, die Deuts...",Der viel beschworene Erfndergeist in Deutschla...,beschworene erfndergeist deutschland obersten ...,innovationen ermöglichen deutschland voranbringen,beschworen Erfndergeist Deutschland oberer Pri...,Innovation ermöglichen Deutschland voranbringen,Innovation ermöglichen Deutschland voranbringen,7,431,438,411 - Technology and Infrastructure
8,SPD,2. Wir kämpfen für Made in Germany 2.0.,"Wir wollen eine stabile, breit aufgestellte un...","Unsere Unternehmen müssen sicher sein, dass si...",unternehmen müssen sicher klimaneutralität wei...,stabile breit aufgestellte zukunftsfähige wirt...,Unternehmen müssen sicher Klimaneutralität wei...,stabil breit aufgestellt zukunftsfähig Wirtschaft,stabil breit aufgestellt zukunftsfähig Wirtschaft,9,284,293,408 - Economic Goals
...,...,...,...,...,...,...,...,...,...,...,...,...,...
618,Die Linke,17. Für eine gerechte Einwanderungsgesellschaf...,Niemand flieht freiwillig,Das Chaos an den europäischen Grenzen ist das ...,chaos europäischen grenzen ergebnis politikver...,niemand flieht freiwillig,Chaos europäisch Grenze Ergebnis Politikversag...,niemand fliehen freiwillig,niemand fliehen freiwillig,3,330,333,201 - Freedom and Human Rights
619,Die Linke,17. Für eine gerechte Einwanderungsgesellschaf...,Wir sind eine Einwanderungsgesellschaft – und ...,Die großen Herausforderungen von mehr Personal...,großen herausforderungen mehr personal gesundh...,einwanderungsgesellschaft müssen,groß Herausforderung mehr Personal Gesundheit ...,Einwanderungsgesellschaft müssen,Einwanderungsgesellschaft müssen,10,586,596,607 - Multiculturalism: Positive
622,Die Linke,19. Medien und Kultur für eine plurale Gesells...,Kultur – vielfältig und für alle zugänglich,Der Zugang zu Kultur soll nicht vom Geldbeutel...,zugang kultur geldbeutel abhängen kunst kultur...,kultur vielfältig zugänglich,Zugang Kultur Geldbeutel abhängen Kunst Kultur...,Kultur vielfältig zugänglich,Kultur vielfältig zugänglich,7,254,261,503 - Equality: Positive
623,Die Linke,19. Medien und Kultur für eine plurale Gesells...,Sport ist kein Luxus,Sport ist für alle da. Dafür müssen die Zugang...,sport dafür müssen zugangsbedingungen verbesse...,sport luxus,Sport dafür müssen Zugangsbedingung verbessern...,Sport Luxus,Sport Luxus,4,136,140,503 - Equality: Positive


In [37]:
labled_theses_df

,These,These_no_stopwords,These_lemmatized,These_unique_words,CDU / CSU,GRÜNE,SPD,AfD,Die Linke,FDP,Predicted_Class
0,Alle Beschäftigten sollen bereits nach 40 Beit...,beschäftigten sollen bereits 40 beitragsjahren...,beschäftigter sollen bereits 40 beitragsjahr a...,beschäftigter sollen bereits 40 beitragsjahr a...,stimme nicht zu,stimme nicht zu,stimme nicht zu,stimme nicht zu,stimme zu,stimme nicht zu,504 - Welfare State Expansion
1,Alle Bürgerinnen und Bürger sollen in gesetzli...,bürgerinnen bürger sollen gesetzlichen kranken...,bürgerinn Bürger sollen gesetzlich Krankenkass...,bürgerinn Bürger sollen gesetzlich Krankenkass...,stimme nicht zu,stimme zu,stimme zu,stimme nicht zu,stimme zu,stimme nicht zu,504 - Welfare State Expansion
2,An Bahnhöfen soll die Bundespolizei Software z...,bahnhöfen bundespolizei software automatisiert...,bahnhöfen Bundespolizei Software automatisiert...,bahnhöfen Bundespolizei Software automatisiert...,stimme zu,stimme nicht zu,stimme nicht zu,stimme zu,stimme nicht zu,stimme nicht zu,605 - Law and Order: Positive
3,Asylsuchende sollen in Deutschland sofort nach...,asylsuchende sollen deutschland sofort antrags...,asylsuchender sollen Deutschland sofort Antrag...,asylsuchender sollen Deutschland sofort Antrag...,stimme nicht zu,stimme zu,stimme zu,stimme nicht zu,stimme zu,neutral,602 - National Way of Life: Negative
4,"Asylsuchende, die über einen anderen EU-Staat ...",asylsuchende eingereist sollen deutschen grenz...,asylsuchender einreisen sollen deutsch Grenze ...,asylsuchender einreisen sollen deutsch Grenze ...,stimme zu,stimme nicht zu,stimme nicht zu,stimme zu,stimme nicht zu,stimme zu,705 - Underprivileged Minority Groups
5,Auf allen Autobahnen soll ein generelles Tempo...,autobahnen generelles tempolimit gelten,autobahn Generelle tempolimit gelten,autobahn Generelle tempolimit gelten,stimme nicht zu,stimme zu,stimme zu,stimme nicht zu,stimme zu,stimme nicht zu,411 - Technology and Infrastructure
6,Aus Deutschland sollen weiterhin Rüstungsgüter...,deutschland sollen weiterhin rüstungsgüter isr...,Deutschland sollen weiterhin rüstungsgüter Isr...,Deutschland sollen weiterhin rüstungsgüter Isr...,stimme zu,stimme zu,stimme zu,neutral,stimme nicht zu,stimme zu,107 - Internationalism: Positive
7,Bei Neuvermietungen sollen die Mietpreise weit...,neuvermietungen sollen mietpreise weiterhin ge...,Neuvermietung sollen mietpreise weiterhin gese...,Neuvermietung sollen mietpreise weiterhin gese...,stimme zu,stimme zu,stimme zu,stimme nicht zu,stimme zu,stimme nicht zu,412 - Controlled Economy
8,Bei der Besteuerung von Einkommen soll der Spi...,besteuerung einkommen spitzensteuersatz angehoben,Besteuerung Einkommen Spitzensteuersatz anheben,Besteuerung Einkommen Spitzensteuersatz anheben,stimme nicht zu,stimme zu,stimme zu,stimme nicht zu,stimme zu,stimme nicht zu,503 - Equality: Positive
9,Beim Ausbau der Verkehrsinfrastruktur soll die...,beim ausbau verkehrsinfrastruktur schiene vorr...,bei Ausbau verkehrsinfrastruktur Schiene Vorra...,bei Ausbau verkehrsinfrastruktur Schiene Vorra...,stimme nicht zu,stimme zu,stimme zu,stimme nicht zu,stimme zu,stimme nicht zu,411 - Technology and Infrastructure


In [11]:
lable_colums_list = labled_theses_df.columns[1:].tolist()

replace_dict = {
    'spd'   :   'SPD', 
    'cdu'   :   'CDU / CSU', 
    'gruene':   'GRÜNE', 
    'afd'   :   'AfD',
    'fdp'   :   'FDP', 
    'linke' :   'Die Linke'
}

all_party_textes_df['Party'] = all_party_textes_df['Party'].replace(replace_dict)
del replace_dict

In [12]:
# Load tokenizer and model
model_name = "mmmaurer/sbert-hashtag-german-politicians-2021"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)



def get_embedding(text:str, pooling_method:str = 'mean'):
    # Tokenize the input
    inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt")

    # Get model output (embeddings)
    outputs = model(**inputs)

    # Extract the embeddings from the model output
    match pooling_method:
        case 'mean':
            return outputs.last_hidden_state.mean(dim=1).squeeze().detach().numpy()
        case 'max':
            return outputs.last_hidden_state.max(dim=1).values.squeeze().detach().numpy()
        case _:
            raise ValueError(f'The used pooling method ({pooling_method}) is not supported. Use "mean" or "max".')




In [ ]:
def similarity_func(all_party_textes_df: pd.DataFrame, 
                    labled_theses_df: pd.DataFrame, 
                    lable_text_column: str, 
                    party_text_column: str,
                    min_similarity_percentage:float = 0.6,
                    plot: bool = False, 
                    pooling_method: str = 'mean'):
    
    # Adjazenzmatrix einlesen
    matrix = pd.read_parquet('adjazenzmatrix.parquet')

    # Liste der Parteien und Thesen
    all_party_textes_df['Chapter'] = all_party_textes_df['Chapter'].fillna('-')
    parties_list = all_party_textes_df['Party'].unique().tolist()
    predicted_theses_df = copy.deepcopy(labled_theses_df)
    
    labled_text_list = labled_theses_df[lable_text_column].tolist()
    predicted_theses_dfs_dict = {}

    # Schleife über die Parteien
    for party_str in parties_list:
        # Filtern der Daten für die aktuelle Partei
        party_df = all_party_textes_df[all_party_textes_df['Party'] == party_str]

        # Entfernen von Zeilen mit NaN-Werten
        party_df = party_df.dropna()

        # Extrahieren der Thesen für die aktuelle Partei
        party_text_list = party_df[party_text_column].to_list()

        # Erstellung eines DataFrames zur Berechnung der Ähnlichkeiten
        similarity_df = pd.DataFrame(columns=['party_text'] + labled_text_list)

        # Berechnung der Ähnlichkeit für jede Partei-These
        for party_text in party_text_list:
            party_text_embedding = get_embedding(party_text, pooling_method=pooling_method)
            row = [party_text]

            # Verwende nur die ersten 3 Zeichen der Predicted_Class der Partei
            party_domain_num = party_df[party_df[party_text_column]==party_text]['Predicted_Class'].str[0:3].values[0]
            # Berechnung der Kosinusähnlichkeit für jede labelte These
            for label_text in labled_text_list:
                # Verwende nur die ersten 3 Zeichen der Predicted_Class der These
                label_domain_num = labled_theses_df[labled_theses_df[lable_text_column] == label_text]['Predicted_Class'].str[0:3].values[0]
                label_text_embedding = get_embedding(label_text, pooling_method=pooling_method)
                similarity_score = cosine_similarity(party_text_embedding.reshape(1, -1), label_text_embedding.reshape(1, -1))[0][0]

                # Wenn die Predicted_Class der Partei und der These übereinstimmt, verwenden wir die Kosinusähnlichkeit
                if matrix.loc[label_domain_num, party_domain_num] == 1:
                    similarity_score = similarity_score  if similarity_score > min_similarity_percentage else 0
                    
                # Wenn keine Übereinstimmung gefunden wird, suchen wir in der Adjazenzmatrix nach dem Gegenteil
                elif matrix.loc[label_domain_num, party_domain_num] == -1:
                    similarity_score = -similarity_score if similarity_score > min_similarity_percentage else 0

                else:
                    similarity_score = 0  # Keine Übereinstimmung und kein Gegenteil, setze Ähnlichkeit auf 0
                        
                row.append(similarity_score)

            similarity_df.loc[len(similarity_df)] = row

        if plot:
            # Visualisierung der Ähnlichkeiten als Heatmap
            plt.figure(figsize=(18, 18))
            sns.heatmap(similarity_df[labled_text_list],
                        cmap='RdYlGn',
                        annot=False,
                        vmax=1,
                        vmin=0,
                        yticklabels=party_df["These"])
            plt.xticks(rotation=45, ha="right")
            plt.title(f"Thesen-Heatmap: {party_str}")
            plt.show()

        # Maximale Ähnlichkeit für jede Partei
        predicted_theses_dfs_dict[party_str] = similarity_df
        max_index = similarity_df[labled_text_list].abs().idxmax()
        predicted_theses_df[party_str] = similarity_df.loc[max_index, labled_text_list]

        # Neue Spalte für Predicted_Class der Partei-These
        predicted_theses_df[f'Predicted_Class_{party_str}'] = party_df[party_df[party_text_column] == party_text_list[0]]['Predicted_Class'].values[0]

    return predicted_theses_dfs_dict, predicted_theses_df


In [ ]:
lable_text_column = 'These_lemmatized'
party_text_column = 'Text_lemmatized'
predicted_theses_dfs_dict, predicted_theses_df = similarity_func(all_party_textes_df,labled_theses_df,lable_text_column,party_text_column, plot=False)

In [38]:
predicted_theses_df

,These,These_no_stopwords,These_lemmatized,These_unique_words,CDU / CSU,GRÜNE,SPD,AfD,Die Linke,FDP,Predicted_Class
0,Alle Beschäftigten sollen bereits nach 40 Beit...,beschäftigten sollen bereits 40 beitragsjahren...,beschäftigter sollen bereits 40 beitragsjahr a...,beschäftigter sollen bereits 40 beitragsjahr a...,0.962512,0.938636,0.899766,0.965782,0.397355,0.962983,504 - Welfare State Expansion
1,Alle Bürgerinnen und Bürger sollen in gesetzli...,bürgerinnen bürger sollen gesetzlichen kranken...,bürgerinn Bürger sollen gesetzlich Krankenkass...,bürgerinn Bürger sollen gesetzlich Krankenkass...,0.970416,0.955666,0.910952,0.969547,0.456947,0.954334,504 - Welfare State Expansion
2,An Bahnhöfen soll die Bundespolizei Software z...,bahnhöfen bundespolizei software automatisiert...,bahnhöfen Bundespolizei Software automatisiert...,bahnhöfen Bundespolizei Software automatisiert...,0.886722,0.822612,0.837915,0.819789,0.211306,0.837813,605 - Law and Order: Positive
3,Asylsuchende sollen in Deutschland sofort nach...,asylsuchende sollen deutschland sofort antrags...,asylsuchender sollen Deutschland sofort Antrag...,asylsuchender sollen Deutschland sofort Antrag...,0.976928,0.917511,0.924830,0.952186,0.113132,0.946283,602 - National Way of Life: Negative
4,"Asylsuchende, die über einen anderen EU-Staat ...",asylsuchende eingereist sollen deutschen grenz...,asylsuchender einreisen sollen deutsch Grenze ...,asylsuchender einreisen sollen deutsch Grenze ...,0.954652,0.870280,0.897872,0.901188,0.550789,0.705962,705 - Underprivileged Minority Groups
5,Auf allen Autobahnen soll ein generelles Tempo...,autobahnen generelles tempolimit gelten,autobahn Generelle tempolimit gelten,autobahn Generelle tempolimit gelten,0.864220,0.847844,0.644067,0.835338,0.779577,0.881656,411 - Technology and Infrastructure
6,Aus Deutschland sollen weiterhin Rüstungsgüter...,deutschland sollen weiterhin rüstungsgüter isr...,Deutschland sollen weiterhin rüstungsgüter Isr...,Deutschland sollen weiterhin rüstungsgüter Isr...,0.974566,0.966524,0.419544,0.786348,0.165666,0.965274,107 - Internationalism: Positive
7,Bei Neuvermietungen sollen die Mietpreise weit...,neuvermietungen sollen mietpreise weiterhin ge...,Neuvermietung sollen mietpreise weiterhin gese...,Neuvermietung sollen mietpreise weiterhin gese...,0.960311,0.930100,0.683911,0.945936,0.857836,0.963424,412 - Controlled Economy
8,Bei der Besteuerung von Einkommen soll der Spi...,besteuerung einkommen spitzensteuersatz angehoben,Besteuerung Einkommen Spitzensteuersatz anheben,Besteuerung Einkommen Spitzensteuersatz anheben,0.957139,0.964250,0.893859,0.966535,0.370453,0.956623,503 - Equality: Positive
9,Beim Ausbau der Verkehrsinfrastruktur soll die...,beim ausbau verkehrsinfrastruktur schiene vorr...,bei Ausbau verkehrsinfrastruktur Schiene Vorra...,bei Ausbau verkehrsinfrastruktur Schiene Vorra...,0.927553,0.918009,0.712146,0.867325,0.888973,0.899092,411 - Technology and Infrastructure
